In [0]:
%sql
-- 1. Check which catalog our session is currently using by default
SELECT current_catalog() AS active_catalog;

active_catalog
workspace


In [0]:
%sql
-- Set our active catalog so we don't have to type it every time
USE CATALOG workspace;

-- 2. Create the three Medallion Architecture schemas
CREATE SCHEMA IF NOT EXISTS bronze
COMMENT 'Raw ingestion layer: exact copies of source data in Delta format with audit metadata';

CREATE SCHEMA IF NOT EXISTS silver
COMMENT 'Cleaned, deduplicated, Conformed, and schema-enforced enterprise data';

CREATE SCHEMA IF NOT EXISTS gold
COMMENT 'Business-level aggregates and dimensional models (Star Schema) ready for BI and analytics';

-- 3. Create a Managed Volume inside the Bronze schema to hold our raw Kaggle CSV files
CREATE VOLUME IF NOT EXISTS bronze.raw_landing
COMMENT 'Landing zone for raw Olist E-Commerce CSV files uploaded from Kaggle';

In [0]:
# 1. Define the Unity Catalog Volume path as a string variable
volume_path = "/Volumes/workspace/bronze/raw_landing/"

# 2. Use Databricks Utilities to list the files in the Volume
raw_files = dbutils.fs.ls(volume_path)

# 3. Display the results in a formatted table
display(raw_files)

path,name,size,modificationTime
dbfs:/Volumes/workspace/bronze/raw_landing/olist_customers_dataset.csv,olist_customers_dataset.csv,9033957,1792768711000
dbfs:/Volumes/workspace/bronze/raw_landing/olist_geolocation_dataset.csv,olist_geolocation_dataset.csv,61273883,1792776684000
dbfs:/Volumes/workspace/bronze/raw_landing/olist_order_items_dataset.csv,olist_order_items_dataset.csv,15438671,1792769291000
dbfs:/Volumes/workspace/bronze/raw_landing/olist_order_payments_dataset.csv,olist_order_payments_dataset.csv,5777138,1792765033000
dbfs:/Volumes/workspace/bronze/raw_landing/olist_order_reviews_dataset.csv,olist_order_reviews_dataset.csv,14451670,1792769221000
dbfs:/Volumes/workspace/bronze/raw_landing/olist_orders_dataset.csv,olist_orders_dataset.csv,17654914,1792769401000
dbfs:/Volumes/workspace/bronze/raw_landing/olist_products_dataset.csv,olist_products_dataset.csv,2379446,1792764920000
dbfs:/Volumes/workspace/bronze/raw_landing/olist_sellers_dataset.csv,olist_sellers_dataset.csv,174703,1792764845000
dbfs:/Volumes/workspace/bronze/raw_landing/product_category_name_translation.csv,product_category_name_translation.csv,2613,1792764836000


In [0]:
from pyspark.sql.functions import current_timestamp, col

# 1. Read the raw CSV into a DataFrame
raw_customers_df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(f"{volume_path}olist_customers_dataset.csv")

# 2. Add audit metadata columns using the Unity Catalog _metadata feature
bronze_customers_df = raw_customers_df.withColumn("ingestion_timestamp", current_timestamp()).withColumn("source_file", col("_metadata.file_path"))

# 3. Write the DataFrame as a managed Delta table in the bronze schema
bronze_customers_df.write.format("delta").mode("overwrite").saveAsTable("workspace.bronze.customers")

# 4. View the resulting Delta table
display(spark.table("workspace.bronze.customers").limit(5))

customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,ingestion_timestamp,source_file
06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP,2026-10-23T17:47:55.561Z,dbfs:/Volumes/workspace/bronze/raw_landing/olist_customers_dataset.csv
18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP,2026-10-23T17:47:55.561Z,dbfs:/Volumes/workspace/bronze/raw_landing/olist_customers_dataset.csv
4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP,2026-10-23T17:47:55.561Z,dbfs:/Volumes/workspace/bronze/raw_landing/olist_customers_dataset.csv
b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP,2026-10-23T17:47:55.561Z,dbfs:/Volumes/workspace/bronze/raw_landing/olist_customers_dataset.csv
4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP,2026-10-23T17:47:55.561Z,dbfs:/Volumes/workspace/bronze/raw_landing/olist_customers_dataset.csv


In [0]:
from pyspark.sql.functions import current_timestamp, col

# 1. Fetch the list of all files currently in our landing volume
raw_files = dbutils.fs.ls(volume_path)

# 2. Loop through each file in the volume
for file in raw_files:
    file_name = file.name
    
    # 3. Only process CSV files (safety check)
    if file_name.endswith(".csv"):
        
        # 4. Clean up the table name (e.g., 'olist_orders_dataset.csv' -> 'orders_dataset')
        table_name = file_name.replace("olist_", "").replace(".csv", "")
        print(f"Ingesting {file_name} into workspace.bronze.{table_name}...")
        
        # 5. Read the raw CSV dynamically
        df = spark.read.format("csv") \
            .option("header", "true") \
            .option("inferSchema", "true") \
            .load(f"{volume_path}{file_name}")
            
        # 6. Append our modern Unity Catalog audit columns
        df_bronze = df.withColumn("ingestion_timestamp", current_timestamp()) \
                      .withColumn("source_file", col("_metadata.file_path"))
                      
        # 7. Write to Bronze tier as a managed Delta table
        df_bronze.write.format("delta") \
                 .mode("overwrite") \
                 .saveAsTable(f"workspace.bronze.{table_name}")
                 
print("All Bronze tables successfully ingested!")

Ingesting olist_customers_dataset.csv into workspace.bronze.customers_dataset...
Ingesting olist_geolocation_dataset.csv into workspace.bronze.geolocation_dataset...
Ingesting olist_order_items_dataset.csv into workspace.bronze.order_items_dataset...
Ingesting olist_order_payments_dataset.csv into workspace.bronze.order_payments_dataset...
Ingesting olist_order_reviews_dataset.csv into workspace.bronze.order_reviews_dataset...
Ingesting olist_orders_dataset.csv into workspace.bronze.orders_dataset...
Ingesting olist_products_dataset.csv into workspace.bronze.products_dataset...
Ingesting olist_sellers_dataset.csv into workspace.bronze.sellers_dataset...
Ingesting product_category_name_translation.csv into workspace.bronze.product_category_name_translation...
All Bronze tables successfully ingested!
